In [1]:
import os

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["JAX_ENABLE_X64"] = "true"

from pathlib import Path
import shutil
import sys
import json

import ase.io
from ase.visualize import view
import numpy as onp

from msmjax.calculators import (
    set_up_msm_params,
    MSMParams,
    check_cutoffs_and_get_actual_spacings,
)
import msmjax

parentpath_charged_lj_utils = (
    Path(msmjax.__file__).resolve().parents[2] / "examples" / "charged_lj_md"
)

sys.path.append(str(parentpath_charged_lj_utils))
path_charged_lj_utils = parentpath_charged_lj_utils / "utils_charged_lj"

from utils_charged_lj.energy_model import SIGMA_ANGSTROM
from utils_charged_lj.helpers import read_logfile_ase

In [2]:
BASEINDIR = Path("NVT/")

# Get target pressure for NPT simulation from NVT warmup simulation

In [3]:
atoms_original = ase.io.read(BASEINDIR / "out" / "md.traj", index=-1)
atoms_original.wrap()
n_particles = len(atoms_original)
sidelength_original = atoms_original.cell.cellpar()[0]
cell_original = onp.asarray(atoms_original.get_cell())
msm_params_original = MSMParams.load_json(BASEINDIR / "msm_params.json")

In [4]:
view(atoms_original)

<Popen: returncode: None args: ['/home/florian/anaconda3/envs/msmjax_rev/bin...>

# Set up MSM parameters with a "safety margin" built in for cell compression/expansion

It is a good idea to monitor during/after the simulation if the cell stayed within these limits.

In [5]:
def get_params_with_safety_margin(
    sidelength_lo: float,
    sidelength_hi: float,
    lower_spacing_thresh: float,
    level_zero_cutoff: float,
    n_dim: int = 3,
    **kwargs,
):
    # Find the smallest legal (see
    # msmjax.bspline.gridops.find_spacings_and_max_level_periodic) number of
    # grid points that still results in a grid spacing less than or equal to
    # lower_spacing_thresh in the most expanded state (side_length_hi):
    exponent_one = onp.ceil(
        onp.log2(sidelength_hi / lower_spacing_thresh)
    ).astype(int)
    exponent_three = onp.ceil(
        onp.log2(sidelength_hi / lower_spacing_thresh / 3)
    ).astype(int)
    n_one = 2**exponent_one
    n_three = 3 * 2**exponent_three
    n_gridpoints = min(n_one, n_three)

    # Get the corresponding grid spacing in the most compressed state and use
    # it to determine the stencil size required to cover the cutoff.
    spacing_lo = sidelength_lo / n_gridpoints
    cell_lo = onp.eye(n_dim) * sidelength_lo

    params = set_up_msm_params(
        cell=cell_lo,
        level_one_spacings=spacing_lo,
        level_zero_cutoff=level_zero_cutoff,
        **kwargs,
    )

    # The actual resulting safe limits may be wider than the originally
    # required ones, since grid points can only be added in discrete steps.
    # => find these actual, widest possible, limits and return them as well.
    sidelength_lo_actual, sidelength_hi_actual = get_cell_size_limits(
        params, lower_spacing_thresh=lower_spacing_thresh
    )
    # Double-check correctness:
    eps = 1.0e-2
    check_cutoffs_and_get_actual_spacings(
        onp.eye(n_dim) * sidelength_lo_actual + eps, params
    )
    check_cutoffs_and_get_actual_spacings(
        onp.eye(n_dim) * sidelength_hi_actual - eps, params
    )

    return params, sidelength_lo_actual, sidelength_hi_actual


def _get_cell_size_lims(
    cutoff_lvl_0: float,
    n_gridpoints_lvl_1: int,
    stencil_size_lvl_1: int,
    lower_spacing_thresh: float,
):
    sidelength_lo_stencils = (2 * n_gridpoints_lvl_1 * cutoff_lvl_0) / (
        stencil_size_lvl_1 + 1
    )
    sidelength_lo_mic = cutoff_lvl_0 / 2

    sidelength_hi = n_gridpoints_lvl_1 * lower_spacing_thresh

    return max(sidelength_lo_stencils, sidelength_lo_mic), sidelength_hi


def get_cell_size_limits(
    params: MSMParams, lower_spacing_thresh: float
) -> tuple[float, float]:
    sidelength_lo, sidelength_hi = _get_cell_size_lims(
        cutoff_lvl_0=params.cutoffs[0],
        n_gridpoints_lvl_1=params.grid_shapes[1][0],
        stencil_size_lvl_1=params.stencil_extents_from_center[1][0],
        lower_spacing_thresh=lower_spacing_thresh,
    )
    return sidelength_lo, sidelength_hi

In [6]:
# Somewhat arbitrary choice; it is a good idea to monitor the sidelength
# during/after a run to check if it stayed within the predefined limits.
minimum_safe_limits = (0.95, 1.25)

sidelength_lo = minimum_safe_limits[0] * sidelength_original
sidelength_hi = minimum_safe_limits[1] * sidelength_original

common_setup_kwargs = {
    "p": 4,
    "pbc": (True, True, True),
    "cell_mode": "ortho",
    "dynamic_cell": True,
}

msm_params_safety, sidelength_lo, sidelength_hi = (
    get_params_with_safety_margin(
        sidelength_lo=sidelength_lo,
        sidelength_hi=sidelength_hi,
        lower_spacing_thresh=SIGMA_ANGSTROM,
        level_zero_cutoff=msm_params_original.cutoffs[0],
        **common_setup_kwargs,
    )
)

# Additionally we want to make sure the grid spacing is never larger than
# the average interparticle distance:
assert msm_params_safety.grid_shapes[1][0] >= n_particles ** (1 / 3)

# Since grid points can be added only in discrete steps, the actual safe limits
# can be wider than the ones requested during setup:
sidelength_lo_actual, sidelength_hi_actual = get_cell_size_limits(
    msm_params_safety, lower_spacing_thresh=SIGMA_ANGSTROM
)

# Get target pressure and initial temperature of the temperature ramp

In [7]:
# Get a reasonable pressure for the NPT simulation by averaging over
# the second half of the NVT trajectory:
logfile = BASEINDIR / "out" / "md.log"
loaded_log = read_logfile_ase(logfile, stress=True)
target_pressure_GPa = -loaded_log["stress_GPa"][
    len(loaded_log["stress_GPa"]) // 2 :, :3
].mean()

In [8]:
# Get the temperature that the preceding NVT warmup simulation was run at:
with open(BASEINDIR / "md_setup_info.json", "r") as f:
    temp_K_start = json.load(f)["temp_K"]

# Construct temperature ramp and write input files for NPT simulation

In [9]:
ENSEMBLE = "NPT"
RUNDIR = Path("NPT-temperature-ramp/")

TEMP_K_END = 1200.0
N_RAMP_SEGMENTS = 30

SIMTIME_PS = 3000.0
TIMESTEP_FS = 1.0
LOGINTERVAL_FS = 500.0

FILENAME_MD_SCRIPT = "run_md_ase.py"
FILENAME_INIT_STRUCT = "initial_structure.xyz"
DIRNAME_OUT = "out/"
FILENAME_MSM_PARAMS = "msm_params.json"

range_of_temps = onp.linspace(temp_K_start, TEMP_K_END, N_RAMP_SEGMENTS)
range_of_temps_str = "(" + " ".join([str(t) for t in range_of_temps]) + ")"
simtime_ps_per_segment = SIMTIME_PS / N_RAMP_SEGMENTS

In [10]:
scripttext_definitions = f"""#!/bin/bash

FILENAME_INIT_STRUCT="{FILENAME_INIT_STRUCT}"
BASEOUTDIR="{DIRNAME_OUT}"

FILENAME_MSM_PARAMS="{FILENAME_MSM_PARAMS}"
TIMESTEP_FS={TIMESTEP_FS}
LOGINTERVAL_FS={LOGINTERVAL_FS}
PRESSURE_GPA={target_pressure_GPa}
ENSEMBLE="NPT"

N_TEMPS={N_RAMP_SEGMENTS}
RANGE_OF_TEMPS={range_of_temps_str}
SIMTIME_PS_PER_SEGMENT={simtime_ps_per_segment}
"""

scripttext_body = r"""
INPUTFILE="${FILENAME_INIT_STRUCT}"
SEGMENTDIR="${BASEOUTDIR}/0/"

for ((i=0;i<N_TEMPS;i++)); do
    CURRENT_TEMP="${RANGE_OF_TEMPS[$i]}"

    echo "- Running temperature segment ${i} out of $((N_TEMPS-1))"
    echo "- T = ${CURRENT_TEMP} K"
    echo "- Reading initial_structure from: ${INPUTFILE}"
    echo "- Writing output to:              ${SEGMENTDIR}"

    python run_md_ase.py \
        --inputstruct $INPUTFILE \
        --msm_params $FILENAME_MSM_PARAMS \
        --outdir $SEGMENTDIR \
        --timestep_fs $TIMESTEP_FS \
        --loginterval_fs $LOGINTERVAL_FS \
        --simtime_ps $SIMTIME_PS_PER_SEGMENT \
        --ensemble $ENSEMBLE \
        --temp_K $CURRENT_TEMP \
        --pressure_GPa $PRESSURE_GPA

    INPUTFILE="${SEGMENTDIR}/md.traj"
    SEGMENTDIR="${BASEOUTDIR}/$((i+1))/"

    echo
done
"""

scripttext = scripttext_definitions + scripttext_body

In [11]:
md_setup_info = {
    "simtime_ps": SIMTIME_PS,
    "timestep_fs": TIMESTEP_FS,
    "loginterval_fs": LOGINTERVAL_FS,
    "ensemble": ENSEMBLE,
    "range_of_temps_K": range_of_temps.tolist(),
    "pressure_GPa": target_pressure_GPa,
    "safe_sidelength_limits": [sidelength_lo_actual, sidelength_hi_actual],
}

RUNDIR.mkdir()

ase.io.write(RUNDIR / FILENAME_INIT_STRUCT, atoms_original)
msm_params_safety.save_json(RUNDIR / FILENAME_MSM_PARAMS, indent=2)
with open(RUNDIR / "md_setup_info.json", "w") as f:
    json.dump(md_setup_info, f, indent=2)
shutil.copy(path_charged_lj_utils / FILENAME_MD_SCRIPT, RUNDIR)

with open(RUNDIR / "runscript.sh", "w") as f:
    f.write(scripttext)